# Finetuning LVH

Running `LVH_detection/main.py` in this notebook.


## (Main) LVH Detection

In [ ]:
import os
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
import tqdm
from sklearn import metrics

# Locate LVH_detection/ regardless of the kernel's working directory
LVH_DIR = next(
    p for p in (Path.cwd(), Path.cwd() / "LVH_detection", Path.cwd() / "echo-severe-as" / "LVH_detection")
    if (p / "dataset.py").exists()
)
sys.path.insert(0, str(LVH_DIR))

from dataset import EchoDatasetLVHDetection
from utils import evaluate, seed_worker, set_seed, train, validate

In [ ]:
data_dir = LVH_DIR.parent.parent / "data" 
output_dir = "output"
clip_len = 16
sampling_rate = 1
num_clips = 4
model_name = "3dresnet18"
rand_init = False
n_gpu = 1
max_epochs = 10
patience = 1e4
lr = 1e-4
batch_size = 10
label_smoothing = 0.0
use_class_weights = False
dropout_fc = False
augment = False
lpft = ""
ssl = str(LVH_DIR.parent / "ssl_pretraining/output/model-2026-1002")
frac = 1.0
weight_averaging = False
n_TTA = 0
seed = 0

In [ ]:
MODEL_NAME = model_name
MODEL_NAME += f"_frac-{frac}" if frac != 1.0 else ""
MODEL_NAME += "_pretr" if not rand_init else "_rand"
MODEL_NAME += f"_ssl-{ssl.split('/')[-1]}" if ssl != "" else ""
MODEL_NAME += "_aug" if augment else ""
MODEL_NAME += f"_clip-len-{clip_len}-stride-{sampling_rate}"
MODEL_NAME += f"_num-clips-{num_clips}"
MODEL_NAME += "_cw" if use_class_weights else ""
MODEL_NAME += f"_lr-{lr}"
MODEL_NAME += f"_{max_epochs}ep"
MODEL_NAME += f"_patience-{patience}" if patience != 1e4 else ""
MODEL_NAME += f"_bs-{batch_size}"
MODEL_NAME += f"_ls-{label_smoothing}" if label_smoothing != 0.0 else ""
MODEL_NAME += f"_drp-0.25" if dropout_fc else ""
MODEL_NAME += f"_weight-avg" if weight_averaging else ""
MODEL_NAME += f"_seed-{seed}" if seed != 0 else ""

MODEL_NAME += "_val-auc"

MODEL_NAME

In [ ]:
# Create output directory for model (and delete if already exists)
# inside echo-severe-as/ (the parent of LVH_detection/), whatever the kernel's cwd
output_dir = LVH_DIR.parent / "output"
output_dir.mkdir(exist_ok=True)

model_dir = output_dir / MODEL_NAME
if os.path.isdir(model_dir):
    shutil.rmtree(model_dir)
os.mkdir(model_dir)

model_dir

In [ ]:
# Set all seeds for reproducibility
set_seed(seed)

# Set device (main.py hardcodes cuda:0; fall back so this also runs on a Mac)
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

In [ ]:
# Create model
if model_name == "3dresnet18":
    weights = torchvision.models.video.R3D_18_Weights.DEFAULT if not rand_init else None
    model = torchvision.models.video.r3d_18(weights=weights)

    if dropout_fc:
        model.fc = torch.nn.Sequential(  # type: ignore
            torch.nn.Linear(512, 1), torch.nn.Dropout(0.25)
        )

        if lpft != "":
            # Linear probing fine-tuning (LPFT) weights are stored in a dict under the key "weights"
            weights = dict(torch.load(lpft, map_location="cpu")["weights"])

            model.fc[0].weight.data = weights["fc.0.weight"]
            model.fc[0].bias.data = weights["fc.0.bias"]

    else:
        model.fc = torch.nn.Linear(512, 1)
        if lpft != "":
            # Linear probing fine-tuning (LPFT) weights are stored in a dict under the key "weights"
            weights = dict(torch.load(lpft, map_location="cpu")["weights"])

            model.fc.weight.data = weights["fc.weight"]
            model.fc.bias.data = weights["fc.bias"]

    if ssl != "":
        checkpoints = [f for f in os.listdir(ssl) if f.endswith(".pt")]
        idx = np.argmax([int(f.split(".")[0].split("-")[1]) for f in checkpoints])
        weights = torch.load(os.path.join(ssl, checkpoints[idx]), map_location="cpu")[
            "weights"
        ]
        weights = {
            k.replace("encoder.", ""): v for k, v in weights.items() if "encoder" in k
        }

        model.load_state_dict(weights, strict=False)


model = model.to(device)

In [ ]:
train_dataset = EchoDatasetLVHDetection(
    data_dir=data_dir,
    split="train",
    clip_len=clip_len,
    sampling_rate=sampling_rate,
    num_clips=num_clips,
    augment=augment,
    kinetics=(ssl == "") and (not rand_init),
)

val_dataset = EchoDatasetLVHDetection(
    data_dir=data_dir,
    split="val",
    clip_len=clip_len,
    sampling_rate=sampling_rate,
    num_clips=num_clips,
    kinetics=(ssl == "") and (not rand_init),
)

test_dataset = EchoDatasetLVHDetection(
    data_dir=data_dir,
    split="test",
    clip_len=clip_len,
    sampling_rate=sampling_rate,
    num_clips=num_clips,
    kinetics=(ssl == "") and (not rand_init),
)


In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=n_gpu * batch_size,
    shuffle=True,
    num_workers=8,
    worker_init_fn=seed_worker,
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=n_gpu * batch_size,
    shuffle=False,
    num_workers=8,
    worker_init_fn=seed_worker,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=n_gpu * batch_size,
    shuffle=False,
    num_workers=8,
)

In [ ]:
# Create csv documenting training history
history = pd.DataFrame(
    columns=[
        "epoch",
        "phase",
        "loss",
        "auroc",
        "aupr",
        "acc",
        "b_acc",
        "mcc",
        "precision",
        "recall",
        "f1",
    ]
)
history.to_csv(os.path.join(model_dir, "history.csv"), index=False)

In [ ]:
loss_fxn = torch.nn.BCEWithLogitsLoss()
eval_loss_fxn = torch.nn.BCELoss()

In [ ]:
# Create optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Train with early stopping
epoch = 1
early_stopping_dict = {"best_auroc": 0.0, "epochs_no_improve": 0}
best_model_wts = None

In [ ]:
while epoch <= max_epochs and early_stopping_dict["epochs_no_improve"] < patience:
    history = train(
        model=model,
        device=device,
        loss_fxn=loss_fxn,
        optimizer=optimizer,
        data_loader=train_loader,
        history=history,
        epoch=epoch,
        model_dir=model_dir,
    )
    history, early_stopping_dict, best_model_wts = validate(
        model=model,
        device=device,
        loss_fxn=eval_loss_fxn,
        optimizer=optimizer,
        data_loader=val_loader,
        history=history,
        epoch=epoch,
        model_dir=model_dir,
        early_stopping_dict=early_stopping_dict,
        best_model_wts=best_model_wts,
    )

    epoch += 1

In [ ]:
# # Evaluate on test set
evaluate(
    model=model,
    device=device,
    loss_fxn=eval_loss_fxn,
    data_loader=test_loader,
    split="test",
    classes=test_dataset.CLASSES,
    history=history,
    model_dir=model_dir,
    weights=best_model_wts,
)
